# Stroke Risk Prediction — Precision Tuning
This notebook trains a compact Random Forest using the five input fields. It selects the classification threshold on validation data, then evaluates once on a separate test set. Precision often rises when the threshold rises, but recall usually falls.

In [ ]:
!pip -q install pandas scikit-learn joblib

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.lower().endswith('.csv')]
if not csv_files:
    raise ValueError('Upload the dataset CSV.')
CSV_PATH = csv_files[0]
print('Using:', CSV_PATH)

In [ ]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
print(df.columns.tolist())
display(df.head())

In [ ]:
import json, joblib, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

FEATURES=['age','hypertension','heart_disease','avg_glucose_level','bmi']
TARGET='stroke'
missing=[c for c in FEATURES+[TARGET] if c not in df.columns]
if missing: raise ValueError(f'Missing required columns: {missing}')
data=df[FEATURES+[TARGET]].copy()
for col in FEATURES+[TARGET]: data[col]=pd.to_numeric(data[col],errors='coerce')
data=data.dropna(subset=[TARGET])
data[TARGET]=data[TARGET].astype(int)
X,y=data[FEATURES],data[TARGET]

X_trainval,X_test,y_trainval,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=.20,random_state=42,stratify=y_trainval)

model=Pipeline([
 ('imputer',SimpleImputer(strategy='median')),
 ('classifier',RandomForestClassifier(n_estimators=35,max_depth=9,min_samples_leaf=5,
  class_weight='balanced_subsample',random_state=42,n_jobs=1))
])
model.fit(X_train,y_train)
val_scores=model.predict_proba(X_val)[:,1]

# Select threshold on validation only: maximize precision with recall >= 0.50, if possible.
candidates=np.unique(np.r_[np.arange(.10,.96,.025),.5,.975])
rows=[]
for t in candidates:
 p=(val_scores>=t).astype(int)
 prec=precision_score(y_val,p,zero_division=0)
 rec=recall_score(y_val,p,zero_division=0)
 f1=f1_score(y_val,p,zero_division=0)
 rows.append((float(t),float(prec),float(rec),float(f1),int(p.sum())))
eligible=[r for r in rows if r[2]>=.50 and r[4]>0]
chosen=max(eligible,key=lambda r:(r[1],r[3],r[0])) if eligible else max(rows,key=lambda r:(r[3],r[1]))
threshold=chosen[0]
print('Selected threshold:',threshold)
print('Validation precision:',round(chosen[1],4),'recall:',round(chosen[2],4))

In [ ]:
test_scores=model.predict_proba(X_test)[:,1]
def report_at(t):
 p=(test_scores>=t).astype(int)
 return {
  'threshold':round(float(t),4),
  'accuracy':round(float(accuracy_score(y_test,p)),4),
  'precision':round(float(precision_score(y_test,p,zero_division=0)),4),
  'recall':round(float(recall_score(y_test,p,zero_division=0)),4),
  'f1':round(float(f1_score(y_test,p,zero_division=0)),4),
  'confusion_matrix':confusion_matrix(y_test,p).tolist()
 }
metrics={
 'rows':int(len(data)),
 'train_rows':int(len(X_train)),
 'validation_rows':int(len(X_val)),
 'test_rows':int(len(X_test)),
 'stroke_positive_count':int(y.sum()),
 'stroke_positive_rate_percent':round(float(y.mean()*100),2),
 'validation_threshold_precision':round(chosen[1],4),
 'validation_threshold_recall':round(chosen[2],4),
 'test_at_default_threshold':report_at(.5),
 'test_at_tuned_threshold':report_at(threshold)
}
print(json.dumps(metrics,indent=2))
print('Tuned-threshold classification report:')
print(classification_report(y_test,(test_scores>=threshold).astype(int),zero_division=0))

In [ ]:
joblib.dump(model,'stroke_model.joblib',compress=3)
with open('threshold.json','w') as f: json.dump({'threshold':float(threshold)},f,indent=2)
with open('training_metrics.json','w') as f: json.dump(metrics,f,indent=2)
from google.colab import files
files.download('stroke_model.joblib')
files.download('threshold.json')
files.download('training_metrics.json')

## Caution
The threshold is selected on validation data, not on the final test set. Higher precision commonly means lower recall. This is a classroom project, not a validated medical diagnostic tool.